<a href="https://colab.research.google.com/github/molluIdontknow/ELE_Algorithm/blob/%EA%B9%80%EC%8B%9C%EC%9A%B0/ELEalgo_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""to"""
import pandas as pd
from sklearn.datasets import fetch_california_housing

# 1. 캘리포니아 하우싱 데이터셋 로드
housing = fetch_california_housing(as_frame=True)
df = housing.frame
CSV_FILEPATH = "california_housing.csv"
df.to_csv(CSV_FILEPATH, index=False)

# 기본 실행 코드

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# ==========================================
# 1. 데이터셋 클래스 (CSV 파일 읽기)
# ==========================================
class BatteryCSVDataset(Dataset):
    def __init__(self, csv_file, feature_cols, target_col):
        # pandas로 CSV를 읽고 PyTorch Tensor로 변환
        df = pd.read_csv(csv_file)
        self.x = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.y = torch.tensor(df[target_col].values, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# ==========================================
# MLP 모델 클래스 (가장 간편한 구조)
# ==========================================
class SOHPredictorMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # nn.Sequential을 사용하면 forward 함수를 아주 깔끔하게 짤 수 있습니다.
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)

# 모델 실행

In [ ]:
CSV_FILENAME = "california_housing.csv"
FEATURES = ["MedInc", "HouseAge", "AveRooms", "Population", "AveOccup", "Latitude", "Longitude"] # 피처 5개 예시
TARGET = "MedHouseVal"
batch_size=100
# 1) 데이터 로더 준비 (Batch Size 2개씩 분할)
dataset = BatteryCSVDataset(CSV_FILENAME, feature_cols=FEATURES, target_col=TARGET)
dataloader = DataLoader(dataset, batch_size, shuffle=True)

# 2) 모델, 손실함수(MSE), 최적화 기법(Adam) 선언
model = SOHPredictorMLP(input_dim=len(FEATURES))
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("\n=== 모델 학습 시작 ===")

# 3) 간단한 학습 루프 (예: 5 Epoch)
for epoch in range(1, 51):
  total_loss = 0
  for batch_x, batch_y in dataloader:

      optimizer.zero_grad()             # 1. 기울기 초기화
      pred_y = model(batch_x)           # 2. SOH 예측
      loss = criterion(pred_y, batch_y) # 3. 오차(Loss) 계산
      loss.backward()                   # 4. 역전파
      optimizer.step()                  # 5. 가중치 업데이트

      total_loss += loss.item()

  print(f"Epoch [{epoch}/50] | 평균 Loss: {total_loss/len(dataloader):.4f}")

print("\n 파이프라인 연동 테스트 완료!")

model.eval() # 평가 모드 전환

# 전체 데이터셋 크기(len(dataset)) 중에서 무작위로 10개 인덱스 선택
random_indices = np.random.choice(len(dataset), size=10, replace=False)

# 선택된 무작위 인덱스의 데이터를 리스트로 모아 Tensor로 결합
sample_x_list = [dataset[i][0] for i in random_indices]
sample_y_list = [dataset[i][1] for i in random_indices]

sample_x_subset = torch.stack(sample_x_list)
sample_y_subset = torch.stack(sample_y_list)

# 추론 실행
with torch.no_grad():
    preds = model(sample_x_subset)

# Tensor -> NumPy 1차원 배열로 변환
actuals = sample_y_subset.numpy().flatten()
predictions = preds.numpy().flatten()

# 절대 오차 및 오차율(%) 연산: |실제값 - 예측값| / 실제값 * 100
abs_error = np.abs(actuals - predictions)
error_rate_pct = (abs_error / np.maximum(actuals, 1e-8)) * 100

# 보기 쉽게 Pandas DataFrame으로 정리
result_df = pd.DataFrame({
    "원본 Index": random_indices,  # 몇 번째 행 데이터를 뽑았는지 확인용
    "실제값 (Actual)": np.round(actuals, 4),
    "예측값 (Pred)": np.round(predictions, 4),
    "절대 오차": np.round(abs_error, 4),
    "오차율 (%)": np.round(error_rate_pct, 2)
})

print("\n=== 📊 무작위 추출 10개 샘플 예측 결과 비교 ===")
print(result_df.to_string(index=False))


=== 모델 학습 시작 ===
Epoch [1/50] | 평균 Loss: 200.7687
Epoch [2/50] | 평균 Loss: 0.9088
Epoch [3/50] | 평균 Loss: 0.6801
Epoch [4/50] | 평균 Loss: 0.6740
Epoch [5/50] | 평균 Loss: 0.5967
Epoch [6/50] | 평균 Loss: 0.5926
Epoch [7/50] | 평균 Loss: 0.6026
Epoch [8/50] | 평균 Loss: 0.7239
Epoch [9/50] | 평균 Loss: 0.6039
Epoch [10/50] | 평균 Loss: 0.5866
Epoch [11/50] | 평균 Loss: 0.5755
Epoch [12/50] | 평균 Loss: 0.6149
Epoch [13/50] | 평균 Loss: 0.6107
Epoch [14/50] | 평균 Loss: 0.6087
Epoch [15/50] | 평균 Loss: 0.6378
Epoch [16/50] | 평균 Loss: 0.6082
Epoch [17/50] | 평균 Loss: 0.5918
Epoch [18/50] | 평균 Loss: 0.5639
Epoch [19/50] | 평균 Loss: 0.5878
Epoch [20/50] | 평균 Loss: 0.5806
Epoch [21/50] | 평균 Loss: 0.6215
Epoch [22/50] | 평균 Loss: 0.5843
Epoch [23/50] | 평균 Loss: 0.5703
Epoch [24/50] | 평균 Loss: 0.5788
Epoch [25/50] | 평균 Loss: 0.5817
Epoch [26/50] | 평균 Loss: 0.5901
Epoch [27/50] | 평균 Loss: 0.5516
Epoch [28/50] | 평균 Loss: 0.5615
Epoch [29/50] | 평균 Loss: 0.5625
Epoch [30/50] | 평균 Loss: 0.5554
Epoch [31/50] | 평균 Loss: 0.56